In [1]:
pip install pandas numpy matplotlib seaborn nltk


Active code page: 1252
Note: you may need to restart the kernel to use updated packages.


In [2]:
import nltk
nltk.download('stopwords')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\satya\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [7]:
# ===============================================
# 🌾 Project Samarth - Notebook 03
# Phase 2: Intelligent Q&A System (Final Fixed)
# ===============================================

# ✅ Step 1: Import Libraries
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

# ✅ Step 2: Load Integrated Dataset
data_path = "../hybrid_dataset/merged_agri_rainfall.csv"
df = pd.read_csv(data_path)

df.columns = df.columns.str.lower().str.strip()
df["crop_year"] = pd.to_numeric(df["crop_year"], errors="coerce")

print("✅ Dataset Loaded Successfully!")
print(f"Rows: {len(df)}, Columns: {len(df.columns)}")
print("\n📊 Columns:", df.columns.tolist())
print("\n🏛️ States:", df['state_name'].dropna().unique().tolist())

# ===============================================
# 🧠 Step 3: Improved NLP Query Parser
# ===============================================

def parse_query(query: str):
    """
    Improved NLP parser that cleanly extracts states, crop, years, and metrics.
    Example:
      'Compare rainfall and rice production in Andaman and Nicobar Islands and Andhra Pradesh for the last 5 years'
    """
    query = query.lower().strip()

    # Extract crop
    crop_match = re.findall(r"\b(rice|wheat|maize|sugarcane|banana|cotton)\b", query)

    # Extract metrics
    metrics = []
    if "rainfall" in query: metrics.append("rainfall")
    if "production" in query: metrics.append("production")

    # Extract year info
    year_match = re.search(r"last (\d+)", query)
    years = int(year_match.group(1)) if year_match else 5

    # Extract state names (clean multiple cases)
    state_part = re.search(r"in (.*)", query)
    states = []
    if state_part:
        # Break by 'and', ',', or 'with'
        parts = re.split(r"\band\b|,|with", state_part.group(1))
        for p in parts:
            p = p.strip()
            # Stop reading after phrases like "for the last"
            if "for the last" in p:
                break
            if p:
                states.append(p.strip())

    return {
        "states": states,
        "crop": crop_match[0] if crop_match else None,
        "years": years,
        "metrics": metrics
    }

# ===============================================
# ⚙️ Step 4: Query Execution Logic
# ===============================================

def run_query(parsed_query: dict):
    """Perform analysis using parsed query information."""
    if df.empty:
        return {"error": "Dataset not found or empty."}

    states = [s.lower() for s in parsed_query.get("states", [])]
    crop = parsed_query.get("crop")
    years = parsed_query.get("years", 5)
    metrics = parsed_query.get("metrics", [])

    filtered = df.copy()

    if states:
        filtered = filtered[filtered["state_name"].str.lower().isin(states)]
    if crop:
        filtered = filtered[filtered["crop"].str.lower() == crop]

    # Handle missing crop_year safely
    if "crop_year" in filtered.columns and not filtered.empty:
        latest_year = filtered["crop_year"].max()
        if pd.notna(latest_year):
            latest_year = int(latest_year)
            start_year = latest_year - years + 1
            filtered = filtered[(filtered["crop_year"] >= start_year) & (filtered["crop_year"] <= latest_year)]

    if filtered.empty:
        return {"message": "No matching records found for your query."}

    result = {"states": states, "crop": crop, "years": years}

    # Rainfall Analysis
    if "rainfall" in metrics:
        rain_cols = [c for c in ["annual", "jjas", "jf", "mam", "ond"] if c in filtered.columns]
        if rain_cols:
            filtered["avg_rainfall"] = filtered[rain_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)
            rainfall_summary = (
                filtered.groupby("state_name")["avg_rainfall"].mean().round(2).to_dict()
            )
            result["rainfall_summary"] = rainfall_summary

    # Production Analysis
    if "production" in metrics and "production_" in filtered.columns:
        prod_summary = (
            filtered.groupby("state_name")["production_"].sum().round(2).to_dict()
        )
        result["production_summary"] = prod_summary

    return result

# ===============================================
# 🗣️ Step 5: Test with a Query
# ===============================================

example_query = "Compare rainfall and rice production in Andaman and Nicobar Islands and Andhra Pradesh for the last 5 years"

parsed = parse_query(example_query)
print("🔍 Parsed Query:", parsed)

result = run_query(parsed)
print("\n📊 Q&A Result Summary:\n", result)

# ===============================================
# ✅ Step 6: Display Final Answer
# ===============================================

if "message" in result:
    print("ℹ️", result["message"])
elif "rainfall_summary" in result or "production_summary" in result:
    print(f"\n📊 Analysis for {', '.join(parsed['states'])} — Crop: {parsed['crop'].title() if parsed['crop'] else 'All'}")

    if "rainfall_summary" in result:
        print("\n🌧️ Average Rainfall (mm):")
        for s, v in result["rainfall_summary"].items():
            print(f" • {s.title()}: {v} mm")

    if "production_summary" in result:
        print("\n🌾 Total Production (tonnes):")
        for s, v in result["production_summary"].items():
            print(f" • {s.title()}: {int(v)} tonnes")

    print("\n📚 Data Source: Government Open Data Portal (data.gov.in)")
    print("Developed for Project Samarth — Integrating Agriculture & Climate Data 🌦️🌾")

print("\n✅ Notebook Execution Completed Successfully!")


✅ Dataset Loaded Successfully!
Rows: 203, Columns: 25

📊 Columns: ['state_name', 'district_name', 'crop_year', 'season', 'crop', 'area_', 'production_', 'subdivision', 'jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec', 'annual', 'jf', 'mam', 'jjas', 'ond']

🏛️ States: ['andaman and nicobar islands']
🔍 Parsed Query: {'states': ['andaman', 'nicobar islands'], 'crop': 'rice', 'years': 5, 'metrics': ['rainfall', 'production']}

📊 Q&A Result Summary:
 {'message': 'No matching records found for your query.'}
ℹ️ No matching records found for your query.

✅ Notebook Execution Completed Successfully!


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\satya\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [6]:
print("Available States in merged dataset:", df['state_name'].unique().tolist())
print("Available Crops:", df['crop'].unique().tolist()[:10])


Available States in merged dataset: ['andaman and nicobar islands']
Available Crops: ['Arecanut', 'Other Kharif pulses', 'Rice', 'Banana', 'Cashewnut', 'Coconut', 'Dry ginger', 'Sugarcane', 'Sweet potato', 'Tapioca']
